In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

os.environ['GROQ_API_KEY']=os.getenv("GROQ_API_KEY")



In [2]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model_name="qwen/qwen3.8-27b",streaming=True
)

In [3]:
result=llm.invoke("Who is virat kohli")
result.content

'**Virat Kohli** is an Indian professional cricketer and the current captain of the **India national cricket team**. Widely regarded as one of the greatest batsmen in the history of the sport, he is known for his aggressive yet technically sound batting style.\n\n### Key Highlights:\n- **Role**: Batsman\n- **Teams**:\n  - **National**: India (all formats: Test, ODI, T20I)\n  - **Domestic**: Delhi\n  - **International Franchise**: Royal Challengers Bengaluru (RCB) in the Indian Premier League (IPL)\n- **Achievements**:\n  - Holds numerous records, including the most centuries in One Day Internationals (ODIs) and Test matches.\n  - Known for his consistent run-scoring and ability to perform under pressure.\n  - Has been a key player in India’s major tournament successes, including the 2011 Cricket World Cup (ODI) and 2023 T20 World Cup.\n- **Leadership**:\n  - Served as captain of the Indian ODI and T20I teams from 2017 to 2022.\n  - Currently captains the Indian Test team (since 2024) a

In [4]:
%pip install -U pypdf langchain-community

Note: you may need to restart the kernel to use updated packages.


In [20]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../data/Govt_RAG.pdf")
docs = loader.load()

In [22]:
print("Total pages:", len(docs))


Total pages: 570


In [23]:
docs[100].page_content

'(a)\n (i)\n(ii)\n(iii)\n(iv)\n(v)\n(vi)\n(vii)\n(viii)\n(b)\n (i)\n (ii)\n(iii)\n(iv)\n(v)\n(vi)\n(vii)\n(viii)\n(c)\n (i)\n(ii)\n(iii)\n(iv)\n(a)\nBoards and Commissions can be classified on different bases. These are explained as follows.\nA.   Legal Status\nOn the basis of the legal status, the Boards and Commissions can be classified into three categories,\nviz.\nBoards and Commissions established by the Constitution itself or established by the President\nunder his constitutional authority. These are called ‘constitutional bodies’. They include the\nfollowing:\nElection Commission\nUnion Public Service Commission\nState Public Service Commission\nFinance Commission\nOfficial Language Commission\nBackward Classes Commission\nNational Commission for SCs\nNational Commission for STs\nBoards and Commissions established by special Acts of Parliament. These are called\n‘statutory bodies’. They include the following:\nUniversity Grants Commission\nOil and Natural Gas Commission\nCentral

In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50) 
splitted_docs=splitter.split_documents(docs)

In [44]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# Test the embedding model
vector = embeddings.embed_query(
    "What are governance requirements?"
)

print("Embedding dimensions:", len(vector))

Loading weights: 100%|██████████| 103/103 [00:00<?, ?it/s]


Embedding dimensions: 384


In [46]:
from langchain_community.vectorstores import FAISS

# Create the vector store
vector_store = FAISS.from_documents(
    documents=splitted_docs,
    embedding=embeddings
)


In [47]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

results = retriever.invoke(
    "What are the main governance requirements?"
)

for i, doc in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)


--- Result 1 ---
3.
4.
5.
6.
14th Finance Commission Constituted
20th Law Commission Constituted
Practice Questions on Governance (General Studies—Prelims)
Practice Questions on Governance (General Studies—Mains)
Therefore, it is hoped that this comprehensive volume will be more useful and relevant for these
examinations.
I invite suggestions and comments from the readers of this book. They will be incorporated in the
next edition of the book.
M. L
AXMIKANTH
Metadata: {'producer': 'calibre 1.43.0 [http://calibre-ebook.com]', 'creator': 'calibre 1.43.0 [http://calibre-ebook.com]', 'creationdate': '2015-05-23T06:44:10+00:00', 'author': 'M. Laxmikanth', 'keywords': 'N/A', 'title': 'Governance in India', 'source': '../data/Govt_RAG.pdf', 'total_pages': 570, 'page': 17, 'page_label': '18'}

--- Result 2 ---
Governance Systems (including all inter-department and intra-department systems) and Government
to public (including businesses) systems must adhere to this policy. For legacy systems, 

In [48]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. Initialize the LLM
llm = ChatGroq(
    model_name="qwen/qwen3.8-27b",
    streaming=True
)

# 2. Define the prompt template
prompt_template = """
You are a helpful Governance Assistant.
Answer the user's question using only the provided context.

Instructions:
- Give accurate, clear, and beginner-friendly answers.
- Do not invent facts that are not present in the context.
- If the answer is not available in the context, say:
  "I could not find the answer in the provided documents."
- Use bullet points when appropriate.

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

# 3. Format retrieved documents into context
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 4. Build the RAG chain
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# 5. Ask a question
question = "What are the main governance requirements?"

answer = rag_chain.invoke(question)

print(answer)

Based on the provided context, the main governance requirements and principles include:

*   **Inclusion of Vulnerable Groups:** Decision-making must take into consideration the concerns of the most vulnerable people in society.
*   **Informed and Organized Participation:** Participation requires:
    *   Freedom of association and expression.
    *   An organized civil society.
*   **Rule of Law:** Good governance requires fair legal frameworks that are enforced impartially.
*   **Human Rights Protection:** There must be full protection of human rights, particularly those of minorities.
*   **System Adherence to Policy:**
    *   Governance systems (inter-department and intra-department) and Government-to-public systems (including businesses) must adhere to the applicable policy.
    *   Legacy systems must ensure that their interfaces with other systems adhere to this policy.
    *   Designated bodies responsible for standardization must apply these principles when adopting or evolvi